In [24]:
import os

def configure_aws(access_key_id, secret_access_key, region, profile):
    # Set environment variables for AWS configuration
    os.environ['AWS_ACCESS_KEY_ID'] = access_key_id
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_access_key
    os.environ['AWS_DEFAULT_REGION'] = region

    print("AWS configuration has been set successfully for : ",profile)


id_endpoint_name = "<ENDPOINT>" # EPRID SQLCODER
instance = "llama3 SqlCoder"

id_config_data = {'profile':'<PROFILE>', 
                     'access_key_id' :'<ACCESS_KEY_ID>',
                     'secret_access_key' :'<SECRET_ACCESS_KEY>',
                     'region' :'<REGION>',
                     'eprid_endpoint_name':id_endpoint_name}


# SETUP AWS
configure_aws(access_key_id = id_config_data['access_key_id'], 
              secret_access_key = id_config_data['secret_access_key'], 
              region = id_config_data['region'], 
              profile = id_config_data['profile'])

AWS configuration has been set successfully for :  EPRID


In [25]:
PREFIX_PROMPT = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `{question}`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
- If you are fetching data from a table only then use its columns to filter out the data.
- You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\n"""

PREFIX_PROMPT = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `{question}`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- Do not add table alias if not required in query
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
- If you are fetching data from a table only then use its columns to filter out the data.
- You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\nDDL statements:\n\n"""


SUFFIX_PROMPT = """<|eot_id|><|start_header_id|>assistant<|end_header_id|>
The following SQL query best answers the question `{question}`:
```sql
"""

TABLE_INFO = """DDL statements:

CREATE TABLE "city" (
"City_ID" int,
"Official_Name" text,
"Status" text,
"Area_km_2" real,
"Population" real,
"Census_Ranking" text,
PRIMARY KEY ("City_ID")
);


CREATE TABLE "farm" (
"Farm_ID" int,
"Year" int,
"Total_Horses" real,
"Working_Horses" real,
"Total_Cattle" real,
"Oxen" real,
"Bulls" real,
"Cows" real,
"Pigs" real,
"Sheep_and_Goats" real,
PRIMARY KEY ("Farm_ID")
);

CREATE TABLE "farm_competition" (
"Competition_ID" int,
"Year" int,
"Theme" text,
"Host_city_ID" int,
"Hosts" text,
PRIMARY KEY ("Competition_ID"),
FOREIGN KEY (`Host_city_ID`) REFERENCES `city`(`City_ID`)
);


CREATE TABLE "competition_record" (
"Competition_ID" int,
"Farm_ID" int,
"Rank" int,
PRIMARY KEY ("Competition_ID","Farm_ID"),
FOREIGN KEY (`Competition_ID`) REFERENCES `farm_competition`(`Competition_ID`),
FOREIGN KEY (`Farm_ID`) REFERENCES `farm`(`Farm_ID`)
);"""

prompt = (PREFIX_PROMPT + TABLE_INFO + SUFFIX_PROMPT)
print(prompt)

<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `{question}`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
- If you are fetching data from a table only then use its columns to fil

In [27]:
import json
from langchain.llms.sagemaker_endpoint import LLMContentHandler
import streamlit as st
from langchain import SagemakerEndpoint
from langchain import LLMChain
from langchain.prompts import PromptTemplate

class Llama3ContentHandler(LLMContentHandler):
    content_type = "application/json"
    accepts = "application/json"

    def transform_input(self, prompt: str, model_kwargs: dict) -> bytes:
        self.len_prompt = len(prompt)
        input_dict = {
            "inputs": prompt,
            "parameters": model_kwargs
        }
        input_str = json.dumps(input_dict)
        return input_str.encode('utf-8')

    def transform_output(self, output: bytes) -> str:
        response_json = output.read()
        res = json.loads(response_json)
        if type(res) is list:
            return res[0]['generated_text']
        else:
            return res['generated_text']
 
   
def change_llm(llm_chain, instance, endpoint_name, prompt_template, input_variables=None,
               content_handler=Llama3ContentHandler(), max_new_tokens=1024, repetition_penalty=1.1,
               return_full_text=True, stop=None, temperature=None, top_p=None,region=None):
    if input_variables is None:
        input_variables = ["question"]
    llm_chain['instance'] = instance
    llm_chain['endpoint_name'] = endpoint_name
    prompt = PromptTemplate(
        input_variables=input_variables, template=prompt_template
    )

    llm_chain['prompt_template'] = prompt_template
    llm_chain['prompt'] = prompt
    llm_chain['content_handler'] = content_handler

    if stop:
        model_kwargs = {
            "max_new_tokens": max_new_tokens,
            "top_p": top_p,
            "temperature": temperature,
            "repetition_penalty": repetition_penalty,
            "return_full_text": return_full_text,
            "stop": stop
        }
    else:
        model_kwargs = {
            "max_new_tokens": max_new_tokens,
            "top_p": top_p,
            "temperature": temperature,
            "repetition_penalty": repetition_penalty,
            "return_full_text": return_full_text
        }

    llm = SagemakerEndpoint(
        endpoint_name=llm_chain['endpoint_name'],
        region_name=region,
        model_kwargs=model_kwargs,
        endpoint_kwargs={"CustomAttributes": 'accept_eula=true'},
        content_handler=content_handler
    )
    chain = LLMChain(llm=llm, prompt=prompt, verbose=True)  # Verbose turned on

    # Push all common references to the dict
    llm_chain['llm'] = llm
    llm_chain['chain'] = chain

    return llm_chain

In [28]:
llm_chain = {}

content_handler = Llama3ContentHandler()
return_full_text = False
stop = ""
temperature=0.01
top_p=0.7
max_new_tokens = 1024
return_full_text = False
prompt_template=prompt

llm_chain = change_llm(llm_chain, instance, endpoint_name, prompt_template,
                           input_variables=["question"],
                           content_handler=content_handler, max_new_tokens=max_new_tokens, repetition_penalty=0.95,
                           return_full_text=return_full_text, stop=stop,temperature=temperature,top_p=top_p,region=id_config_data['region'])

In [22]:
qlist = ["How many farms are there?","Count the number of farms.","List the total number of horses on farms in ascending order.","What is the total horses record for each farm, sorted ascending?"]

qlist = ['How many farms are there?',
 'Count the number of farms.',
 'List the total number of horses on farms in ascending order.',
 'What is the total horses record for each farm, sorted ascending?',
 'What are the hosts of competitions whose theme is not "Aliens"?',
 'Return the hosts of competitions for which the theme is not Aliens?',
 'What are the themes of farm competitions sorted by year in ascending order?',
 'Return the themes of farm competitions, sorted by year ascending.',
 'What is the average number of working horses of farms with more than 5000 total number of horses?',
 'Give the average number of working horses on farms with more than 5000 total horses.',
 'What are the maximum and minimum number of cows across all farms.',
 'Return the maximum and minimum number of cows across all farms.',
 'How many different statuses do cities have?',
 'Count the number of different statuses.',
 'List official names of cities in descending order of population.',
 'What are the official names of cities, ordered descending by population?',
 'List the official name and status of the city with the largest population.',
 'What is the official name and status of the city with the most residents?',
 'Show the years and the official names of the host cities of competitions.',
 'Give the years and official names of the cities of each competition.',
 'Show the official names of the cities that have hosted more than one competition.',
 'What are the official names of cities that have hosted more than one competition?',
 'Show the status of the city that has hosted the greatest number of competitions.',
 'What is the status of the city that has hosted the most competitions?',
 'Please show the themes of competitions with host cities having populations larger than 1000.',
 'What are the themes of competitions that have corresponding host cities with more than 1000 residents?',
 'Please show the different statuses of cities and the average population of cities with each status.',
 'What are the statuses and average populations of each city?',
 'Please show the different statuses, ordered by the number of cities that have each.',
 'Return the different statuses of cities, ascending by frequency.',
 'List the most common type of Status across cities.',
 'What is the most common status across all cities?',
 'List the official names of cities that have not held any competition.',
 'What are the official names of cities that have not hosted a farm competition?',
 'Show the status shared by cities with population bigger than 1500 and smaller than 500.',
 'Which statuses correspond to both cities that have a population over 1500 and cities that have a population lower than 500?',
 'Find the official names of cities with population bigger than 1500 or smaller than 500.',
 'What are the official names of cities that have population over 1500 or less than 500?',
 'Show the census ranking of cities whose status are not "Village".',
 'What are the census rankings of cities that do not have the status "Village"?']

sql_correct_list = ['SELECT count(*) FROM farm',
 'SELECT count(*) FROM farm',
 'SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC',
 'SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC',
 "SELECT Hosts FROM farm_competition WHERE Theme !=  'Aliens'",
 "SELECT Hosts FROM farm_competition WHERE Theme !=  'Aliens'",
 'SELECT Theme FROM farm_competition ORDER BY YEAR ASC',
 'SELECT Theme FROM farm_competition ORDER BY YEAR ASC',
 'SELECT avg(Working_Horses) FROM farm WHERE Total_Horses  >  5000',
 'SELECT avg(Working_Horses) FROM farm WHERE Total_Horses  >  5000',
 'SELECT max(Cows) ,  min(Cows) FROM farm',
 'SELECT max(Cows) ,  min(Cows) FROM farm',
 'SELECT count(DISTINCT Status) FROM city',
 'SELECT count(DISTINCT Status) FROM city',
 'SELECT Official_Name FROM city ORDER BY Population DESC',
 'SELECT Official_Name FROM city ORDER BY Population DESC',
 'SELECT Official_Name ,  Status FROM city ORDER BY Population DESC LIMIT 1',
 'SELECT Official_Name ,  Status FROM city ORDER BY Population DESC LIMIT 1',
 'SELECT T2.Year ,  T1.Official_Name FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID',
 'SELECT T2.Year ,  T1.Official_Name FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID',
 'SELECT T1.Official_Name FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID GROUP BY T2.Host_city_ID HAVING COUNT(*)  >  1',
 'SELECT T1.Official_Name FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID GROUP BY T2.Host_city_ID HAVING COUNT(*)  >  1',
 'SELECT T1.Status FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID GROUP BY T2.Host_city_ID ORDER BY COUNT(*) DESC LIMIT 1',
 'SELECT T1.Status FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID GROUP BY T2.Host_city_ID ORDER BY COUNT(*) DESC LIMIT 1',
 'SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID WHERE T1.Population  >  1000',
 'SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID  =  T2.Host_city_ID WHERE T1.Population  >  1000',
 'SELECT Status ,  avg(Population) FROM city GROUP BY Status',
 'SELECT Status ,  avg(Population) FROM city GROUP BY Status',
 'SELECT Status FROM city GROUP BY Status ORDER BY COUNT(*) ASC',
 'SELECT Status FROM city GROUP BY Status ORDER BY COUNT(*) ASC',
 'SELECT Status FROM city GROUP BY Status ORDER BY COUNT(*) DESC LIMIT 1',
 'SELECT Status FROM city GROUP BY Status ORDER BY COUNT(*) DESC LIMIT 1',
 'SELECT Official_Name FROM city WHERE City_ID NOT IN (SELECT Host_city_ID FROM farm_competition)',
 'SELECT Official_Name FROM city WHERE City_ID NOT IN (SELECT Host_city_ID FROM farm_competition)',
 'SELECT Status FROM city WHERE Population  >  1500 INTERSECT SELECT Status FROM city WHERE Population  <  500',
 'SELECT Status FROM city WHERE Population  >  1500 INTERSECT SELECT Status FROM city WHERE Population  <  500',
 'SELECT Official_Name FROM city WHERE Population  >  1500 OR Population  <  500',
 'SELECT Official_Name FROM city WHERE Population  >  1500 OR Population  <  500',
 'SELECT Census_Ranking FROM city WHERE Status !=  "Village"',
 'SELECT Census_Ranking FROM city WHERE Status !=  "Village"']

output = []
for question in qlist:
    sqlcoder_generated_query = llm_chain['chain'].run({"question": question})
    print(sqlcoder_generated_query)
    output.append({"question":question,"sqlquerygenerated":sqlcoder_generated_query})



> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `How many farms are there?`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying column_name + table_name.
- DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the databas


> Finished chain.
SELECT f."Farm_ID", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `What are the hosts of competitions whose theme is not "Aliens"?`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the


> Finished chain.
SELECT fc."Year", fc."Theme" FROM "farm_competition" fc ORDER BY fc."Year" ASC;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `What is the average number of working horses of farms with more than 5000 total number of horses?`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two t


> Finished chain.
SELECT MAX(f.Cows) AS max_cows, MIN(f.Cows) AS min_cows FROM farm f;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `How many different statuses do cities have?`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentiate the same column name by applying col


> Finished chain.
SELECT c.Official_Name, c.Population FROM city c ORDER BY c.Population DESC;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `List the official name and status of the city with the largest population.`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. You can differentia

...


> Finished chain.
WITH HostedCompetitionCounts AS (SELECT c."City_ID", c."Status", COUNT(fc."Competition_ID") AS NumComps FROM "city" c JOIN "farm_competition" fc ON c."City_ID" = fc."Host_city_ID" GROUP BY c."City_ID", c."Status") SELECT hcc."Status" FROM HostedCompetitionCounts hcc ORDER BY hcc.NumComps DESC LIMIT 1;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `Please show the themes of competitions with host cities having populations larger than 1000.`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not aske


> Finished chain.
SELECT c.Status, AVG(c.Population) AS average_population FROM city c GROUP BY c.Status ORDER BY c.Status NULLS LAST;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `Please show the different statuses, ordered by the number of cities that have each.`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specia


> Finished chain.
SELECT c.Status, COUNT(c.Status) AS COUNT FROM city c GROUP BY c.Status ORDER BY COUNT DESC LIMIT 1;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `List the official names of cities that have not held any competition.`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns having same name specially after joining two tables. 


> Finished chain.
SELECT c.Status FROM city c WHERE c.Population > 1500 AND c.Population < 500 GROUP BY c.Status HAVING COUNT(DISTINCT c.Population) = 2;


> Entering new LLMChain chain...
Prompt after formatting:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Generate a SQL query to answer this question: `Find the official names of cities with population bigger than 1500 or smaller than 500.`

### Instructions
- Given an input question, create a syntactically correct query to run, then look at the results of the query and return the answer.
- Never query for all the columns from a specific table, only ask for the relevant columns given the question.
- Only return the columns asked by user, do not give any additional ID column is not asked by user explicitly.
- Do not add any ORDER BY in query if not asked to order by user explicitly.
- If you cannot answer the question with the available database schema, return 'I do not know'
- Make sure that you never return two columns 


> Finished chain.
SELECT c.Census_Ranking FROM city c WHERE c.Status not ilike '%Village%';


In [23]:
for x,correct_sql in zip(output,sql_correct_list):
    print("*"*100)
    print("question : ",x['question'],"\n")
    print("sql generated query : ",x['sqlquerygenerated'],"\n")
    print("correct sql query : ",correct_sql,"\n")
    print("*"*100)

****************************************************************************************************
question :  How many farms are there? 

sql generated query :  SELECT COUNT(*) FROM farm; 

correct sql query :  SELECT count(*) FROM farm 

****************************************************************************************************
****************************************************************************************************
question :  Count the number of farms. 

sql generated query :  SELECT COUNT(f.Farm_ID) FROM farm f; 

correct sql query :  SELECT count(*) FROM farm 

****************************************************************************************************
****************************************************************************************************
question :  List the total number of horses on farms in ascending order. 

sql generated query :  SELECT f.Total_Horses FROM farm f ORDER BY f.Total_Horses ASC; 

correct sql query :  SELECT Total_Horse